# What the Viterbi decoder is worth

The decoder is **post-processing**. The loss is cross-entropy plus soft Dice on raw logits;
no gradient reaches it, and training never sees it. So the same saved logits can be scored
both ways and the difference is the decoding alone - the network is identical in each.

Two passes run after the network:

- **Viterbi** over the four classes. Impossible transitions cost infinity, staying is free,
  an allowed change costs `switch_penalty`.
- **Minimum duration** - a run shorter than its class's floor is absorbed into the longer
  neighbour.

The allowed transitions describe the **label set**, not physiology, so there is one table per
source. Getting that wrong is not subtle: the algorithm table forbids `inhale -> exhale`,
which is a third of every human transition, and Viterbi then bridges it with a one-sample
`unknown` on every breath.

In [ ]:
import sys
from pathlib import Path


def repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'phase').is_dir():
            return candidate
    raise RuntimeError(f'no inhale-exhale-training checkout at or above {start}')


REPO = repo_root(Path.cwd().resolve())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

try:
    import torch  # noqa: F401
except ModuleNotFoundError as missing:
    raise RuntimeError(
        f'{missing.name} is missing - select the inhale-exhale-training kernel') from missing
print(f'repo root {REPO}')

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import Image
from omegaconf import OmegaConf

import report_folds as rf
from models.lightning_module import allowed_for
from phase import figures
from phase.decode import decode, transition_matrix
from phase.labels import PHASES

cfg = OmegaConf.load(REPO / 'parameter' / 'config.yaml')
training = OmegaConf.load(REPO / 'parameter' / 'training' / 'default.yaml')
data_cfg = OmegaConf.load(REPO / 'parameter' / 'data' / 'phases.yaml')
PLOT = OmegaConf.to_container(cfg.plot, resolve=True)
OUT = REPO / 'out' / 'viterbi'

runs = sorted((REPO / 'outputs').glob('*/fold_0/test_logits.npz'),
              key=lambda p: p.stat().st_mtime)
RUN = runs[-1].parent.parent
folds = [rf.load_fold(d) for d in sorted(RUN.iterdir())
         if d.is_dir() and (d / rf.LOGITS_NAME).exists()]
print(f'run {RUN.name}: {len(folds)} folds, '
      f'{len(folds[0]["window_id"])} test windows, '
      f'{len(np.unique(folds[0]["patient"]))} patients')

## The two settings being compared

`raw` is the network's own argmax - no transition table, no duration floors. `viterbi` is the
same logits through both passes.

In [ ]:
decoding = OmegaConf.to_container(training.decoding, resolve=True)
# From the run's own Hydra config, not from `parameter/` - a run may have
# overridden the source on the command line, and the defaults do not say so.
SOURCE = rf.label_source_of(RUN)
allowed = allowed_for(decoding, SOURCE)

COST = transition_matrix(allowed, decoding['switch_penalty'])
MIN_DURATION = decoding['min_duration']
FPS = float(np.median(folds[0]['fps']))
EVENT_IOU = float(training.event_iou)

print(f'label source: {SOURCE}')
print(f'switch penalty {decoding["switch_penalty"]}, floors {MIN_DURATION}\n')
print('allowed transitions:')
for name in PHASES:
    print(f'  {name:8s} -> {", ".join(allowed.get(name, [])) or "(nothing)"}')

### Is the table the right one for these labels?

A transition the labels are full of but the table forbids is the failure that matters -
Viterbi cannot emit it at all, and bridges it instead.

In [ ]:
from phase.building import load_windows, shard_path
from phase.labels import targets_to_spans
import collections

DATASET = Path((RUN / 'fold_0' / 'dataset.txt').read_text().strip())
if not DATASET.is_absolute():
    DATASET = REPO / DATASET
manifest = load_windows(DATASET)

seen = collections.Counter()
for _, row in manifest.iterrows():
    with np.load(shard_path(DATASET, str(row['shard'])), allow_pickle=False) as stored:
        position = int(row['position'])
        start, end = stored['offsets'][position], stored['offsets'][position + 1]
        target = stored['targets'][start:end].astype(int)
    spans = targets_to_spans(target)
    for first, second in zip(spans, spans[1:]):
        seen[(first['phase'], second['phase'])] += 1

total = sum(seen.values())
rows = [{'transition': f'{a} -> {b}', 'share': n / total,
         'allowed': b in allowed.get(a, [])}
        for (a, b), n in seen.most_common()]
table = pd.DataFrame(rows)
forbidden = table[~table.allowed & (table.share > 0.01)]
print(f'{total} span transitions in the labels\n')
print(table.assign(share=(100 * table.share).round(1)).to_string(index=False))
if len(forbidden):
    print(f'\n*** {len(forbidden)} transition(s) over 1% are FORBIDDEN by the table - '
          f'Viterbi will bridge each one ***')
else:
    print('\nno transition above 1% is forbidden - the table fits these labels')

## Scored both ways

The ensemble's logits, decoded and not. Same model, same windows.

In [ ]:
stacked = np.stack([fold['logits'] for fold in folds]).mean(axis=0)
ensemble = {**folds[0], 'logits': stacked}
windows = rf.windows_of(ensemble)

scores = {
    'raw': rf.score_windows(windows, None, None, FPS, EVENT_IOU),
    'viterbi': rf.score_windows(windows, COST, MIN_DURATION, FPS, EVENT_IOU),
}
compare = pd.DataFrame([
    {'metric': m, 'raw': scores['raw'][m], 'viterbi': scores['viterbi'][m],
     'delta': scores['viterbi'][m] - scores['raw'][m]}
    for m in sorted(set(scores['raw']) & set(scores['viterbi']))])
OUT.mkdir(parents=True, exist_ok=True)
compare.to_csv(OUT / 'raw_vs_viterbi.csv', index=False)

headline = ['macro_f1', 'accuracy', *[f'f1_{n}' for n in PHASES],
            *[f'event_f1_{n}' for n in PHASES], 'boundary_mae_sec']
compare[compare.metric.isin(headline)].set_index('metric').loc[
    [m for m in headline if m in set(compare.metric)]].round(3)

In [ ]:
figures.decoding_report(compare, OUT / 'raw_vs_viterbi.png', PLOT)
Image(filename=str(OUT / 'raw_vs_viterbi.png'))

### Reading it

Viterbi trades **per-sample agreement for per-event agreement**. Fragmentation is an
event-level failure - a breath split in two scores well sample by sample and fails as an
event - so the per-event columns are the ones it was built to move. Whether that trade is
worth making depends on which failure matters for the device.

`training.decoding.viterbi: false` turns it off.

## Where it actually changes the answer

Per window, so the effect is not hidden inside a pooled score.

In [ ]:
per_window = []
for index, (logits, truth) in enumerate(windows):
    raw = decode(logits, None, None)
    dec = decode(logits, COST, MIN_DURATION)
    per_window.append({
        'patient': str(ensemble['patient'][index]),
        'window': int(ensemble['window_id'][index]),
        'changed': int((raw != dec).sum()), 'samples': int(truth.size),
        'raw_f1': figures.score(raw, truth), 'viterbi_f1': figures.score(dec, truth)})
per_window = pd.DataFrame(per_window)
per_window['delta'] = per_window.viterbi_f1 - per_window.raw_f1
per_window['changed_pct'] = 100 * per_window.changed / per_window.samples
per_window.to_csv(OUT / 'per_window.csv', index=False)

print(f'windows the decoder altered at all: '
      f'{int((per_window.changed > 0).sum())} of {len(per_window)}')
print(f'median samples changed where it acted: '
      f'{per_window[per_window.changed > 0].changed_pct.median():.1f}%')
print(f'helped {int((per_window.delta > 0).sum())}, '
      f'hurt {int((per_window.delta < 0).sum())}, '
      f'no change {int((per_window.delta == 0).sum())}')

### The windows it changed most

Top row of each pair is the raw argmax, bottom is the same logits decoded.

In [ ]:
def side_by_side(rows, name: str, heading: str):
    panels = []
    for _, row in rows.iterrows():
        index = int(row['index'])
        logits, truth = windows[index]
        values = traces.get((str(ensemble['env'][index]), int(row['window'])))
        if values is None:
            continue
        for label, prediction in (('raw', decode(logits, None, None)),
                                  ('viterbi', decode(logits, COST, MIN_DURATION))):
            panels.append({'values': values['values'], 'reference': truth,
                           'prediction': prediction, 'fps': float(ensemble['fps'][index]),
                           'title': (f"{row['patient']} · window {int(row['window'])} · "
                                     f"signal {values['signal']} · {label} · "
                                     f"F1 {figures.score(prediction, truth):.2f}")})
    if not panels:
        return None
    out = figures.plot_windows(panels, OUT / name, heading, 'labeller', PLOT,
                               prediction_name='model',
                               caption='shading = model · ribbon = labeller')
    return Image(filename=str(out))


traces = rf.traces_for(RUN, ensemble['window_id'], ensemble['env'])
worst = per_window.assign(index=per_window.index).sort_values('delta').head(3)
side_by_side(worst, 'hurt_most.png', 'where the decoder cost the most, raw then decoded')

### And where it helped most

In [ ]:
best = per_window.assign(index=per_window.index).sort_values('delta').tail(3)
side_by_side(best, 'helped_most.png', 'where the decoder gained the most, raw then decoded')